# Training and Evaluation (Aligned with `src/train.py` and `src/evaluate.py`)

This notebook mirrors the production model pipeline exactly:
- load preprocessed train/validation/test artifacts
- train with class weights and early stopping on `val_auc`
- evaluate with ROC AUC, PR AUC, precision, recall, F1, confusion matrix
- save model and metrics to `artifacts/model/`

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
SEED = 42
BATCH_SIZE = 128
EPOCHS = 100
PATIENCE = 5
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
THRESHOLD = 0.5

DATA_DIR = Path("artifacts/data")
MODEL_DIR = Path("artifacts/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
def load_split(path: Path):
    data = np.load(path)
    return data["inputs"].astype(np.float32), data["targets"].astype(np.int32)


train_x, train_y = load_split(DATA_DIR / "train.npz")
val_x, val_y = load_split(DATA_DIR / "validation.npz")
test_x, test_y = load_split(DATA_DIR / "test.npz")

print(train_x.shape, val_x.shape, test_x.shape)

In [ ]:
pos = int(train_y.sum())
neg = len(train_y) - pos
class_weight = {
    0: len(train_y) / (2.0 * max(neg, 1)),
    1: len(train_y) / (2.0 * max(pos, 1)),
}
class_weight

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(train_x.shape[1],)),
    tf.keras.layers.Dense(HIDDEN_SIZE, activation="relu"),
    tf.keras.layers.Dense(HIDDEN_SIZE, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy"), tf.keras.metrics.AUC(name="auc")],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=PATIENCE,
    mode="max",
    restore_best_weights=True,
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(MODEL_DIR / "best_model.keras"),
    monitor="val_auc",
    mode="max",
    save_best_only=True,
)

history = model.fit(
    train_x,
    train_y,
    validation_data=(val_x, val_y),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=[early_stopping, checkpoint],
    verbose=2,
)

In [ ]:
history_path = MODEL_DIR / "history.json"
config_path = MODEL_DIR / "train_config.json"

with history_path.open("w", encoding="utf-8") as fp:
    json.dump(history.history, fp, indent=2)

config = {
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "seed": SEED,
    "class_weight": class_weight,
}
with config_path.open("w", encoding="utf-8") as fp:
    json.dump(config, fp, indent=2)

print(history_path)
print(config_path)

In [ ]:
best_model = tf.keras.models.load_model(MODEL_DIR / "best_model.keras")
probs = best_model.predict(test_x, verbose=0).reshape(-1)
preds = (probs >= THRESHOLD).astype(np.int32)

metrics = {
    "threshold": THRESHOLD,
    "roc_auc": float(roc_auc_score(test_y, probs)),
    "pr_auc": float(average_precision_score(test_y, probs)),
    "precision": float(precision_score(test_y, preds, zero_division=0)),
    "recall": float(recall_score(test_y, preds, zero_division=0)),
    "f1": float(f1_score(test_y, preds, zero_division=0)),
    "confusion_matrix": confusion_matrix(test_y, preds).tolist(),
    "classification_report": classification_report(test_y, preds, zero_division=0, output_dict=True),
}

metrics

In [ ]:
eval_path = MODEL_DIR / "evaluation.json"
with eval_path.open("w", encoding="utf-8") as fp:
    json.dump(metrics, fp, indent=2)

print(
    f"ROC AUC={metrics['roc_auc']:.4f} | PR AUC={metrics['pr_auc']:.4f} | "
    f"Precision={metrics['precision']:.4f} | Recall={metrics['recall']:.4f} | F1={metrics['f1']:.4f}"
)
print("Confusion matrix [ [TN, FP], [FN, TP] ]:", metrics["confusion_matrix"])
print(f"Saved evaluation report to: {eval_path.resolve()}")